<a href="https://colab.research.google.com/github/Juanmipaez/Citacion_1/blob/main/ML1_Mejorado.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==============================
# 1. Subir y descomprimir ML1.zip
# ==============================
from google.colab import files
import zipfile, os
from pathlib import Path

uploaded = files.upload()
zip_path = list(uploaded.keys())[0]

extract_dir = "/content/dataset"
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

path = Path(extract_dir)/"ML1"   # ruta al dataset ya organizado

In [ ]:
# ==============================
# 2. Verificar estructura y conteo de imágenes
# ==============================
for split in ["train", "valid", "test"]:
    five_count = len(list((path/split/"five").glob("*")))
    not_five_count = len(list((path/split/"not_five").glob("*")))
    print(f"{split}: five={five_count}, not_five={not_five_count}")


In [ ]:

# ==============================
# 3. Entrenar modelo con FastAI
# ==============================
!pip install -Uqq fastbook
import fastbook
fastbook.setup_book()

from fastbook import *
from fastai.vision.all import *

# Crear Dataloaders
dls = ImageDataLoaders.from_folder(
    path,
    train="train",
    valid="valid",
    seed=42,
    item_tfms=Resize(224),
    batch_tfms=aug_transforms(mult=1.0)
)

dls.show_batch(max_n=8, figsize=(6,6))

# Crear modelo pre-entrenado
learn = vision_learner(dls, resnet18, metrics=accuracy)
learn.fine_tune(3)


In [ ]:
# ==============================
# 4. Evaluación con matriz de confusión
# ==============================
interp = ClassificationInterpretation.from_learner(learn)
interp.plot_confusion_matrix()


In [ ]:
# ==============================
# 5. Probar con imagen del test
# ==============================
import random

test_images = list((path/"test"/"five").glob("*")) + list((path/"test"/"not_five").glob("*"))
sample_img = random.choice(test_images)

img = PILImage.create(sample_img)
pred, pred_idx, probs = learn.predict(img)

print(f"Imagen: {sample_img.name}")
print(f"Predicción: {pred} ({probs[pred_idx]*100:.2f}% confianza)")
img.show(title=f"Predicción: {pred}, Prob: {probs[pred_idx]:.2f}")